# LeetCode 4006 — Count Valid Prefixes bằng Python

Link đề: https://leetcode.com/problems/count-valid-prefixes/ · Độ khó: **Easy** · Chủ đề: String, Counting, Prefix, Math

Skeleton LeetCode:

```python
class Solution:
    def countValidPrefixes(self, s: str) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một **tiền tố** của chuỗi nhị phân `s` là *hợp lệ* nếu các ký tự của nó có thể **sắp xếp lại** thành chuỗi **xen kẽ** (không có hai ký tự liền kề giống nhau); đếm số tiền tố hợp lệ.
2. Nhận ra **bước đột phá**: sắp xếp lại được thành xen kẽ khi và chỉ khi `|đếm('0') - đếm('1')| <= 1`.
3. Biết duy trì **bộ đếm chạy** (running counters) khi duyệt tiền tố, thay vì tính lại cho từng tiền tố.
4. Viết 4 cách (vét cạn hoán vị, đếm bằng lát cắt, bộ đếm chạy, độ chênh `balance`) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: đếm tiền tố **vốn đã** xen kẽ, đòi hỏi hai số đếm bằng nhau, chỉ kiểm tra tiền tố cuối, đếm cả chuỗi rỗng.

## 2. Đề bài & ví dụ

Cho chuỗi nhị phân `s` (chỉ gồm `'0'` và `'1'`). Một **tiền tố** (prefix) của `s` là `s[:i]` với `1 <= i <= len(s)`. Tiền tố được gọi là **hợp lệ** nếu các ký tự của nó có thể **sắp xếp lại** (hoán vị) thành một chuỗi **xen kẽ**, tức là không có hai ký tự liền kề nào bằng nhau. Trả về số tiền tố hợp lệ.

| Ví dụ | `s` | Output | Giải thích |
|---|---|---|---|
| 1 | `"00101"` | `3` | `"0"` đã xen kẽ; `"001"` sắp lại thành `"010"`; `"00101"` sắp lại thành `"01010"`. Còn `"00"` và `"0010"` thì không |
| 2 | `"101"` | `3` | Mọi tiền tố `"1"`, `"10"`, `"101"` đã xen kẽ |

Constraints: `1 <= len(s) <= 100`; `s` chỉ gồm `'0'` và `'1'`.

Chú ý: ta hỏi "**có thể sắp xếp lại** thành xen kẽ", không phải "đã xen kẽ sẵn".

In [1]:
s = "00101"
for i in range(1, len(s) + 1):
    p = s[:i]
    print(f"tiền tố {p!r}: số 0 = {p.count('0')}, số 1 = {p.count('1')}")

tiền tố '0': số 0 = 1, số 1 = 0
tiền tố '00': số 0 = 2, số 1 = 0
tiền tố '001': số 0 = 2, số 1 = 1
tiền tố '0010': số 0 = 3, số 1 = 1
tiền tố '00101': số 0 = 3, số 1 = 2


## 3. Kiến thức nền

### 3.1. Tiền tố và lát cắt `s[:i]`

`s[:i]` là `i` ký tự đầu. Với `s` dài `n`, các tiền tố khác rỗng là `s[:1], s[:2], ..., s[:n]`: đúng `n` tiền tố. Dùng `range(1, n + 1)` để lấy `i`.

In [2]:
s = "00101"
print([s[:i] for i in range(1, len(s) + 1)])

['0', '00', '001', '0010', '00101']


### 3.2. Đếm ký tự: `str.count`

`p.count('0')` số ký tự `'0'` trong `p`. Với chuỗi nhị phân, `count('1') = len(p) - count('0')`. Gọi `count` cho từng tiền tố tốn `O(n)` mỗi lần, tổng `O(n^2)`; ta sẽ cải thiện bằng bộ đếm chạy.

In [3]:
p = "00101"
print(p.count("0"), p.count("1"), len(p) - p.count("0"))

3 2 2


### 3.3. Chuỗi xen kẽ (alternating)

Chuỗi xen kẽ nhị phân chỉ có hai dạng: `0101...` hoặc `1010...`. Một chuỗi xen kẽ độ dài `L` có số `0` và số `1` **chênh nhau không quá 1**: `L` chẵn -> hai số bằng nhau; `L` lẻ -> chênh đúng 1.

In [4]:
def is_alternating(t):
    return all(a != b for a, b in zip(t, t[1:]))

for t in ["0101", "1010", "010", "0110", "00", "1"]:
    print(t, is_alternating(t), t.count("0"), t.count("1"))

0101 True 2 2
1010 True 2 2
010 True 2 1
0110 False 2 2
00 False 2 0
1 True 0 1


### 3.4. Hoán vị bằng `itertools.permutations`

`permutations("001")` sinh mọi cách xếp (có lặp lại các hoán vị trùng nhau). Dùng làm **trọng tài** vét cạn cho chuỗi ngắn: tồn tại hoán vị xen kẽ hay không.

In [5]:
from itertools import permutations
perms = {"".join(p) for p in permutations("001")}
print(sorted(perms), any(is_alternating(p) for p in perms))

['001', '010', '100'] True


### 3.5. Bộ đếm chạy (running counter)

Khi duyệt `s` từ trái sang phải, giữ `zeros` và `ones` là số `0`, số `1` của tiền tố hiện tại; mỗi ký tự mới chỉ tăng một bộ đếm: `O(1)` mỗi bước. Đây là kỹ thuật nền của mọi bài "đếm trên tiền tố".

In [6]:
zeros = ones = 0
for ch in "00101":
    if ch == "0":
        zeros += 1
    else:
        ones += 1
    print(ch, zeros, ones)

0 1 0
0 2 0
1 2 1
0 3 1
1 3 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chuỗi xen kẽ nhị phân là `0101...` hoặc `1010...`. Nó dùng `0` và `1` **luân phiên**, nên số lượng hai loại **gần bằng nhau**: nếu số `0` lệch quá số `1` hơn `1` đơn vị thì sớm muộn hai số `0` phải đứng cạnh nhau.

Ngược lại, nếu hai số đếm chênh không quá 1, ta **luôn xếp được**: đặt ký tự nhiều hơn (nếu có) trước rồi xen kẽ.

Tính tay `"00101"`:

| tiền tố | số `0` | số `1` | chênh | hợp lệ? |
|---|---|---|---|---|
| `0` | 1 | 0 | 1 | có |
| `00` | 2 | 0 | 2 | không |
| `001` | 2 | 1 | 1 | có |
| `0010` | 3 | 1 | 2 | không |
| `00101` | 3 | 2 | 1 | có |

Có `3` tiền tố hợp lệ. (Chú ý `"001"` không xen kẽ sẵn nhưng sắp lại được.)

In [7]:
s = "00101"
valid = 0
for i in range(1, len(s) + 1):
    p = s[:i]
    z, o = p.count("0"), p.count("1")
    ok = abs(z - o) <= 1
    valid += ok
    print(f"{p:>6}: zeros={z}, ones={o}, |chênh|={abs(z - o)} -> {'hợp lệ' if ok else 'không'}")
print("số tiền tố hợp lệ:", valid)

     0: zeros=1, ones=0, |chênh|=1 -> hợp lệ
    00: zeros=2, ones=0, |chênh|=2 -> không
   001: zeros=2, ones=1, |chênh|=1 -> hợp lệ
  0010: zeros=3, ones=1, |chênh|=2 -> không
 00101: zeros=3, ones=2, |chênh|=1 -> hợp lệ
số tiền tố hợp lệ: 3


## 5. Cách 1 — Brute Force / baseline: thử mọi hoán vị từng tiền tố

Với mỗi tiền tố, thử **mọi hoán vị** của nó và xem có hoán vị nào xen kẽ không. Đúng chắc chắn (theo đúng định nghĩa) nhưng `L!` hoán vị cho tiền tố độ dài `L`: chỉ chạy được với `n <= 8`. Với `n = 100` là bất khả thi: đó là "nút thắt" buộc ta tìm điều kiện toán học.

In [8]:
from itertools import permutations

class SolutionBrute:
    def countValidPrefixes(self, s):
        count = 0
        for i in range(1, len(s) + 1):
            p = s[:i]
            if any(all(a != b for a, b in zip(q, q[1:])) for q in set(permutations(p))):
                count += 1
        return count

print(SolutionBrute().countValidPrefixes("00101"), SolutionBrute().countValidPrefixes("101"))

3 3


**Độ phức tạp:** Time `O(n * n!)` trong trường hợp xấu, Space `O(n!)`. Chỉ dùng cho `n <= 8`.

## 6. Ý tưởng thuật toán / bước đột phá

**Định lý:** một chuỗi nhị phân có thể sắp xếp lại thành chuỗi xen kẽ **khi và chỉ khi** `|zeros - ones| <= 1`.

- *Cần:* chuỗi xen kẽ độ dài `L` có `ceil(L/2)` ký tự loại này và `floor(L/2)` ký tự loại kia, nên chênh `<= 1`.
- *Đủ:* nếu `|zeros - ones| <= 1`, xếp xen kẽ bắt đầu bằng ký tự nhiều hơn (nếu bằng nhau thì bắt đầu bằng ký tự nào cũng được): ví dụ `zeros = 3, ones = 2` -> `01010`.

**Hệ quả thuật toán:** duyệt `s` một lần, duy trì `zeros`, `ones`; sau mỗi ký tự, nếu `|zeros - ones| <= 1` thì cộng một vào đáp án.

**Cách viết gọn:** dùng `balance = zeros - ones` (cộng `+1` cho `'0'`, `-1` cho `'1'`); điều kiện là `-1 <= balance <= 1`.

**Invariant:** sau khi xử lý `i` ký tự, `zeros + ones = i` và `balance` là hiệu hai số đếm của tiền tố độ dài `i`.

## 7. Các cách tiếp theo

### Cách 2 — Đếm bằng lát cắt cho từng tiền tố

Dùng định lý nhưng vẫn gọi `count` trên `s[:i]`: dễ hiểu, `O(n^2)`.

In [9]:
class SolutionSlice:
    def countValidPrefixes(self, s):
        count = 0
        for i in range(1, len(s) + 1):
            p = s[:i]
            if abs(p.count("0") - p.count("1")) <= 1:
                count += 1
        return count

print(SolutionSlice().countValidPrefixes("00101"), SolutionSlice().countValidPrefixes("101"))

3 3


**Độ phức tạp:** Time `O(n^2)` (mỗi `count` là `O(n)`), Space `O(n)` cho lát cắt.

### Cách 3 — Bộ đếm chạy `zeros`, `ones`

Một lượt, `O(1)` mỗi ký tự.

In [10]:
class SolutionRunning:
    def countValidPrefixes(self, s):
        zeros = ones = 0
        count = 0
        for ch in s:
            if ch == "0":
                zeros += 1
            else:
                ones += 1
            if abs(zeros - ones) <= 1:
                count += 1
        return count

print(SolutionRunning().countValidPrefixes("00101"), SolutionRunning().countValidPrefixes("101"))

3 3


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 4 — Một biến `balance`

`balance += 1` với `'0'`, `-= 1` với `'1'`; hợp lệ khi `-1 <= balance <= 1`. Ít biến nhất.

In [11]:
class SolutionBalance:
    def countValidPrefixes(self, s):
        balance = 0
        count = 0
        for ch in s:
            balance += 1 if ch == "0" else -1
            if -1 <= balance <= 1:
                count += 1
        return count

print(SolutionBalance().countValidPrefixes("00101"), SolutionBalance().countValidPrefixes("101"))

3 3


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 4 (`balance`) trên `s = "00101"`:

| ký tự | `balance` sau | trong `[-1, 1]`? | `count` |
|---|---|---|---|
| `0` | 1 | có | 1 |
| `0` | 2 | không | 1 |
| `1` | 1 | có | 2 |
| `0` | 2 | không | 2 |
| `1` | 1 | có | 3 |

Kết quả `3`.

In [12]:
def trace_balance(s):
    balance, count = 0, 0
    print("s =", s)
    for i, ch in enumerate(s, 1):
        balance += 1 if ch == "0" else -1
        ok = -1 <= balance <= 1
        count += ok
        print(f"  tiền tố {s[:i]!r:<8} ký tự={ch}: balance={balance:>2} -> {'hợp lệ' if ok else 'không '} count={count}")
    return count

assert trace_balance("00101") == 3
assert trace_balance("101") == 3
assert trace_balance("0000") == 1

s = 00101
  tiền tố '0'      ký tự=0: balance= 1 -> hợp lệ count=1
  tiền tố '00'     ký tự=0: balance= 2 -> không  count=1
  tiền tố '001'    ký tự=1: balance= 1 -> hợp lệ count=2
  tiền tố '0010'   ký tự=0: balance= 2 -> không  count=2
  tiền tố '00101'  ký tự=1: balance= 1 -> hợp lệ count=3
s = 101
  tiền tố '1'      ký tự=1: balance=-1 -> hợp lệ count=1
  tiền tố '10'     ký tự=0: balance= 0 -> hợp lệ count=2
  tiền tố '101'    ký tự=1: balance=-1 -> hợp lệ count=3
s = 0000
  tiền tố '0'      ký tự=0: balance= 1 -> hợp lệ count=1
  tiền tố '00'     ký tự=0: balance= 2 -> không  count=1
  tiền tố '000'    ký tự=0: balance= 3 -> không  count=1
  tiền tố '0000'   ký tự=0: balance= 4 -> không  count=1


In [13]:
def trace_rearrange(s):
    # xây hoán vị xen kẽ thực sự cho từng tiền tố hợp lệ để minh họa phần "đủ" của định lý
    for i in range(1, len(s) + 1):
        p = s[:i]
        z, o = p.count("0"), p.count("1")
        if abs(z - o) > 1:
            print(f"  {p!r}: không thể ({z} số 0, {o} số 1)")
            continue
        first = "0" if z >= o else "1"
        second = "1" if first == "0" else "0"
        a, b = (z, o) if first == "0" else (o, z)
        out = []
        for k in range(len(p)):
            out.append(first if k % 2 == 0 else second)
        t = "".join(out)
        assert sorted(t) == sorted(p) and is_alternating(t)
        print(f"  {p!r}: xếp lại thành {t!r}")

trace_rearrange("00101")

  '0': xếp lại thành '0'
  '00': không thể (2 số 0, 0 số 1)
  '001': xếp lại thành '010'
  '0010': không thể (3 số 0, 1 số 1)
  '00101': xếp lại thành '01010'


## 9. Code nộp LeetCode

Dùng Cách 4 (một biến `balance`).

In [14]:
from typing import List, Optional


class Solution:
    def countValidPrefixes(self, s: str) -> int:
        balance = 0
        count = 0
        for ch in s:
            balance += 1 if ch == "0" else -1
            if -1 <= balance <= 1:
                count += 1
        return count


print(Solution().countValidPrefixes("00101"), Solution().countValidPrefixes("101"))

3 3


## 10. Vì sao thuật toán đúng

Gọi `z`, `o` là số `0`, số `1` của một tiền tố độ dài `L = z + o`.

1. **Cần.** Trong chuỗi xen kẽ, các ký tự `0` và `1` đứng luân phiên nên sau mỗi ký tự `0` (trừ ký tự cuối) là một `1` và ngược lại; do đó `|z - o| <= 1`.
2. **Đủ.** Giả sử `|z - o| <= 1`. Nếu `z = o`, xếp `0101...01`. Nếu `z = o + 1`, xếp `0101...0` (bắt đầu và kết thúc bằng `0`). Nếu `o = z + 1`, xếp `1010...1`. Mỗi chuỗi dùng đúng `z` số `0` và `o` số `1`, nên là hoán vị của tiền tố, và xen kẽ.
3. **Chuyển sang đếm.** Tiền tố hợp lệ <=> `|z - o| <= 1` <=> `-1 <= balance <= 1` với `balance = z - o`. Mỗi bước cộng/trừ 1 vào `balance` đúng khớp với `z` hoặc `o` tăng 1.
4. **Tính đủ.** Ta xét mọi tiền tố độ dài `1..n` đúng một lần nên đếm không sót không trùng.
5. Cách 1 làm theo định nghĩa, Cách 2, 3 theo định lý nhưng đếm khác, Cách 4 gọn nhất.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s) <= 100`.

- Cách 1 (hoán vị): Time `O(n * n!)`, Space `O(n!)` (chỉ `n <= 8`).
- Cách 2 (lát cắt + `count`): Time `O(n^2)`, Space `O(n)`.
- Cách 3 (hai bộ đếm): Time `O(n)`, Space `O(1)`.
- Cách 4 (`balance`, nộp): Time `O(n)`, Space `O(1)`.

Không thể dưới `O(n)` vì phải đọc từng ký tự của `s`.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [15]:
import random

solutions = [SolutionSlice(), SolutionRunning(), SolutionBalance(), Solution()]

fixed = [
    ("00101", 3), ("101", 3), ("0", 1), ("1", 1), ("00", 1), ("000", 1), ("01", 2), ("0011", 3),
    ("0000", 1), ("0110", 4), ("0101", 4), ("1100", 3), ("0001", 1), ("1" * 100, 1),
    ("01" * 50, 100),
]
for s, expected in fixed:
    for sol in solutions + ([SolutionBrute()] if len(s) <= 7 else []):
        assert sol.countValidPrefixes(s) == expected, (type(sol).__name__, s)
print("test cố định OK:", len(fixed), "ca")

random.seed(4006)
for _ in range(300):                                   # đối chiếu với vét cạn hoán vị (n nhỏ)
    n = random.randint(1, 7)
    s = "".join(random.choice("01") for _ in range(n))
    expected = SolutionBrute().countValidPrefixes(s)
    for sol in solutions:
        assert sol.countValidPrefixes(s) == expected, (type(sol).__name__, s)
print("đối chiếu với vét cạn hoán vị OK (n <= 7)")

for _ in range(500):
    n = random.randint(1, 100)
    p = random.random()
    s = "".join("0" if random.random() < p else "1" for _ in range(n))
    assert len({sol.countValidPrefixes(s) for sol in solutions}) == 1, s
print("test ngẫu nhiên n <= 100 OK")

test cố định OK: 15 ca
đối chiếu với vét cạn hoán vị OK (n <= 7)
test ngẫu nhiên n <= 100 OK


### Test thuộc tính (property test)

(1) Đáp án `1 <= ans <= n` đối với chuỗi một ký tự đầu luôn hợp lệ (độ dài 1 luôn có chênh `1`). (2) Đổi `0 <-> 1` không đổi đáp án. (3) Nối thêm một ký tự không thể làm **giảm** đáp án. (4) Với chuỗi xen kẽ sẵn, đáp án bằng `n`.

In [16]:
random.seed(40060)
for _ in range(500):
    n = random.randint(1, 60)
    s = "".join(random.choice("01") for _ in range(n))
    r = Solution().countValidPrefixes(s)
    assert 1 <= r <= n
    assert Solution().countValidPrefixes(s.translate(str.maketrans("01", "10"))) == r
    assert Solution().countValidPrefixes(s + random.choice("01")) >= r
for n in range(1, 40):
    assert Solution().countValidPrefixes(("01" * 20)[:n]) == n
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đếm tiền tố đã **xen kẽ sẵn** (không cho phép sắp xếp lại)

Đề cho phép hoán vị. Tiền tố `"001"` chưa xen kẽ nhưng sắp lại thành `"010"`.

In [17]:
def wrong_already_alternating(s):
    return sum(1 for i in range(1, len(s) + 1) if all(a != b for a, b in zip(s[:i], s[:i][1:])))   # SAI

print(wrong_already_alternating("00101"), "!=", Solution().countValidPrefixes("00101"))
assert wrong_already_alternating("00101") == 1 and Solution().countValidPrefixes("00101") == 3

1 != 3


### Lỗi 2: đòi hai số đếm **bằng nhau**

Điều kiện đúng là chênh `<= 1`. Đòi bằng nhau loại hết tiền tố độ dài lẻ.

In [18]:
def wrong_equal(s):
    zeros = ones = count = 0
    for ch in s:
        zeros += ch == "0"
        ones += ch == "1"
        if zeros == ones:                                      # SAI: phải |chênh| <= 1
            count += 1
    return count

print(wrong_equal("00101"), "!=", Solution().countValidPrefixes("00101"))
assert wrong_equal("00101") == 0 and wrong_equal("0101") == 2 and Solution().countValidPrefixes("0101") == 4

0 != 3


### Lỗi 3: chỉ kiểm tra **toàn chuỗi** (tiền tố cuối)

Đề đếm mọi tiền tố. Chỉ xét chuỗi đầy đủ cho `0` hoặc `1`.

In [19]:
def wrong_last_only(s):
    return 1 if abs(s.count("0") - s.count("1")) <= 1 else 0     # SAI

print(wrong_last_only("00101"), "!=", Solution().countValidPrefixes("00101"))
assert wrong_last_only("00101") == 1 and Solution().countValidPrefixes("00101") == 3

1 != 3


### Lỗi 4: dùng `<` thay `<=` (hoặc `abs(...) < 1`)

Chênh bằng `1` vẫn hợp lệ (độ dài lẻ). `abs(...) < 1` loại mọi tiền tố độ dài lẻ (chỉ còn các tiền tố có hai số đếm bằng nhau).

In [20]:
def wrong_strict(s):
    balance = count = 0
    for ch in s:
        balance += 1 if ch == "0" else -1
        if abs(balance) < 1:                                     # SAI: phải <= 1
            count += 1
    return count

print(wrong_strict("101"), "!=", Solution().countValidPrefixes("101"))
assert wrong_strict("101") == 1 and Solution().countValidPrefixes("101") == 3

1 != 3


### Lỗi 5: tính cả tiền tố rỗng hoặc bắt đầu từ `0` trong `range`

Tiền tố rỗng (`s[:0]`) không được tính (đề lấy `1 <= i <= n`), dù chênh `0` nên "hợp lệ". Bắt đầu `range(0, n)` thay `range(1, n + 1)` làm tính thêm tiền tố rỗng và bỏ tiền tố đầy đủ.

In [21]:
def wrong_range(s):
    count = 0
    for i in range(0, len(s)):                                   # SAI: gồm tiền tố rỗng, thiếu tiền tố đầy đủ
        p = s[:i]
        if abs(p.count("0") - p.count("1")) <= 1:
            count += 1
    return count

print(wrong_range("0010"), "!=", Solution().countValidPrefixes("0010"))
assert wrong_range("0010") == 3 and Solution().countValidPrefixes("0010") == 2
assert wrong_range("0") == 1                       # với "0": tiền tố rỗng bù cho tiền tố đầy đủ nên tình cờ đúng

3 != 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Sắp xếp lại được thành xen kẽ" -> chỉ cần so **số lượng** hai loại (chênh `<= 1`); không cần thử hoán vị.
- Bài "đếm trên mọi tiền tố" -> **bộ đếm chạy** (`O(1)` mỗi bước) hoặc `balance`; nhớ khởi tạo ngoài vòng lặp.
- `balance = (số 0) - (số 1)` là kỹ thuật quen thuộc (cũng dùng ở "Contiguous Array", "Balanced Strings").
- Kiểm tra mệnh đề toán học nhỏ bằng **vét cạn hoán vị** cho chuỗi ngắn (như ở mục 12).
- Cẩn thận đề "có thể sắp xếp lại" (hoán vị) và "đã" xen kẽ: rất khác nhau.
- Thử ca: `"0"`, `"00"`, chuỗi xen kẽ hoàn toàn, chuỗi toàn một loại.

## 15. Biến thể của bài

1. **Contiguous Array** (525): đoạn con có số `0` = số `1`.
2. **Split a String in Balanced Strings** (1221): chia chuỗi cân bằng `L/R`.
3. **Check if Binary String Has at Most One Segment of Ones** (1784).
4. **Minimum Changes To Make Alternating Binary String** (1758): số thay đổi để thành xen kẽ.
5. Biến thể: đếm **đoạn con** (không chỉ tiền tố) sắp xếp lại được thành xen kẽ (cần đếm bằng hash theo `balance`).
6. Biến thể: ba loại ký tự `a, b, c` (xen kẽ: không hai ký tự giống nhau liền kề): điều kiện `max <= ceil(L/2)`-kiểu (tổng quát: số lớn nhất `<= (L + 1) / 2`).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị | thử mọi hoán vị mỗi tiền tố | `O(n n!)` | `O(n!)` | Chỉ `n <= 7`, làm trọng tài |
| 2. Lát cắt + `count` | `|zeros - ones| <= 1` trên `s[:i]` | `O(n^2)` | `O(n)` | Dễ viết, `n` nhỏ |
| 3. Hai bộ đếm | `zeros`, `ones` chạy | `O(n)` | `O(1)` | Rõ ràng, dễ giải thích |
| 4. `balance` | một biến, `-1 <= balance <= 1` | `O(n)` | `O(1)` | **Nộp bài** |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().countValidPrefixes("1100")` và `Solution().countValidPrefixes("0111")` ra gì?

**Bài 2 (tính tay):** với `s = "010011"`, lập bảng `zeros`, `ones`, `|chênh|` cho từng tiền tố và đếm.

**Bài 3 (điền TODO):** hoàn thành `longest_valid_prefix(s)`: **độ dài lớn nhất** của một tiền tố hợp lệ.

**Bài 4 (tự viết lại):** viết `valid_prefix_lengths(s)` trả về danh sách các độ dài `i` mà tiền tố `s[:i]` hợp lệ.

**Bài 5 (mini challenge):** viết `rearranged(p)` trả về **một** chuỗi xen kẽ là hoán vị của `p` (hoặc `None` nếu không có), rồi kiểm tra bằng `is_alternating`.

In [22]:
# Bài 1: đoán rồi chạy
print(Solution().countValidPrefixes("1100"), Solution().countValidPrefixes("0111"))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def longest_valid_prefix_todo(s):
    # TODO: lưu độ dài i cuối cùng mà |balance| <= 1
    return 0

3 3


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: dùng `balance` (`0 -> +1`, `1 -> -1`). `"1100"`: `-1, -2, -1, 0`. Bài 2: lập bảng 6 dòng.

**Mức 2 (vừa):** Bài 3: duyệt và gán `best = i` mỗi lần hợp lệ. Bài 4: `list.append(i)` thay vì cộng đếm. Bài 5: bắt đầu bằng ký tự nhiều hơn rồi xen kẽ.

**Mức 3 (gần đáp án):** Bài 1: `"1100" -> 3` (độ dài `1, 3, 4`), `"0111" -> 3` (`balance`: `+1, 0, -1, -2` nên hợp lệ ở độ dài `1, 2, 3`). Bài 2: `balance`: `1, 0, 1, 2, 1, 0` -> hợp lệ ở độ dài `1,2,3,5,6` -> `5`. Bài 3: `"00101" -> 5`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().countValidPrefixes("1100") == 3 and Solution().countValidPrefixes("0111") == 3
assert Solution().countValidPrefixes("010011") == 5

# Bài 3
def longest_valid_prefix(s):
    balance, best = 0, 0
    for i, ch in enumerate(s, 1):
        balance += 1 if ch == "0" else -1
        if -1 <= balance <= 1:
            best = i
    return best

assert longest_valid_prefix("00101") == 5 and longest_valid_prefix("0000") == 1 and longest_valid_prefix("0011") == 4

# Bài 4
def valid_prefix_lengths(s):
    balance, res = 0, []
    for i, ch in enumerate(s, 1):
        balance += 1 if ch == "0" else -1
        if -1 <= balance <= 1:
            res.append(i)
    return res

assert valid_prefix_lengths("00101") == [1, 3, 5] and valid_prefix_lengths("010011") == [1, 2, 3, 5, 6]
assert len(valid_prefix_lengths("0111")) == Solution().countValidPrefixes("0111")

# Bài 5
def rearranged(p):
    z, o = p.count("0"), p.count("1")
    if abs(z - o) > 1:
        return None
    first = "0" if z >= o else "1"
    second = "1" if first == "0" else "0"
    return "".join(first if k % 2 == 0 else second for k in range(len(p)))

for p in ["001", "00101", "01", "0", "1", "0011", "000"]:
    t = rearranged(p)
    if abs(p.count("0") - p.count("1")) <= 1:
        assert t is not None and sorted(t) == sorted(p) and is_alternating(t), p
    else:
        assert t is None
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def countValidPrefixes(self, s: str) -> int:
        balance = 0                          # (số '0') - (số '1') của tiền tố hiện tại
        count = 0
        for ch in s:
            balance += 1 if ch == "0" else -1
            if -1 <= balance <= 1:           # sắp xếp lại được thành xen kẽ
                count += 1
        return count
```

Khung chung "đếm trên mọi tiền tố": **cập nhật bộ đếm chạy rồi kiểm tra điều kiện** sau mỗi ký tự.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Điều kiện là `|zeros - ones| <= 1` (chênh **không quá 1**), tính **trên từng tiền tố** (không chỉ cả chuỗi).
- [ ] Không đếm tiền tố rỗng; đã thử `"0"`, `"00"`, `"0101..."`, `"0011"`.